# AI Entrepreneur Coach, MVP

Trait based business idea recommender. This notebook builds the lab MVP step by step:

1. Load API keys
2. Parse the profile PDF (LinkedIn export format)
3. Extract structured profile (skills, experience, industry)
4. Big Five (TIPI) input
5. Hardcoded business idea list
6. Career best fit ranking
7. Output report (working style summary, ranked ideas, rationale, 90 day roadmap)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

assert OPENAI_API_KEY, "OPENAI_API_KEY not found in .env"

print("OPENAI_API_KEY loaded:", bool(OPENAI_API_KEY))
print("COHERE_API_KEY loaded:", bool(COHERE_API_KEY))

## Step 2: Parse the profile PDF

We extract raw text with `pypdf` and split it into these known sections ourselves, no LLM call needed for this step. Cheaper, and each section is inspectable on its own.

Project 3 still needs to handle other CV formats/structures, this parser is written specifically for the LinkedIn export layout.

In [ ]:
from pypdf import PdfReader
import re

PROFILE_PDF_PATH = "input/Profile.pdf"

def fix_line_wrapped_hyphens(text: str) -> str:
    # only join when hyphen attaches to the word directly (no space before it), so a real " - " separator stays untouched
    lines = text.split("\n")
    fixed = []
    i = 0
    while i < len(lines):
        line = lines[i]
        if line.endswith("-") and not line.endswith(" -") and i + 1 < len(lines):
            fixed.append(line + lines[i + 1])
            i += 2
        else:
            fixed.append(line)
            i += 1
    return "\n".join(fixed)

reader = PdfReader(PROFILE_PDF_PATH)
raw_profile_text = "\n".join(page.extract_text() for page in reader.pages)
raw_profile_text = fix_line_wrapped_hyphens(raw_profile_text)

print(raw_profile_text[:500])

In [ ]:
LINKEDIN_SECTIONS = [
    "Contact", "Top Skills", "Languages", "Certifications",
    "Honors-Awards", "Summary", "Experience", "Education",
]

def split_linkedin_sections(text: str) -> dict:
    lines = text.split("\n")
    sections = {}
    current = "header"
    buffer = []
    for line in lines:
        stripped = line.strip()
        if re.match(r"^Page \d+ of \d+$", stripped):
            continue
        if stripped in LINKEDIN_SECTIONS:
            sections[current] = "\n".join(buffer).strip()
            current = stripped
            buffer = []
        else:
            buffer.append(line)
    sections[current] = "\n".join(buffer).strip()
    return sections

def extract_profile_header(sections: dict) -> dict:
    # LinkedIn's export puts name, headline, location right before "Summary" with no header word of its own,
    # so it lands at the tail of whatever sidebar section came last, pull it back out here
    section_names = list(sections.keys())
    if "Summary" not in section_names:
        return {}
    prev_section = section_names[section_names.index("Summary") - 1]
    lines = [l for l in sections[prev_section].split("\n") if l.strip()]
    if len(lines) < 3:
        return {}
    name, headline, location = lines[-3], lines[-2], lines[-1]
    sections[prev_section] = "\n".join(lines[:-3]).strip()
    return {"name": name, "headline": headline, "location": location}

profile_sections = split_linkedin_sections(raw_profile_text)
profile_header = extract_profile_header(profile_sections)

print("profile header:", profile_header)
print()
for section_name, content in profile_sections.items():
    print(f"=== {section_name} ===")
    print(content)
    print()

## Step 3: Extract structured profile

Now one LLM call, on the clean sectioned text (not the raw PDF), constrained to a fixed JSON schema so the output is always usable, not free text. `skills` includes both the explicit "Top Skills" list and skills reasonably implied by the summary/experience text.

In [ ]:
import json
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

extraction_input = f"""Headline: {profile_header.get('headline', '')}
Location: {profile_header.get('location', '')}

Top Skills:
{profile_sections.get('Top Skills', '')}

Summary:
{profile_sections.get('Summary', '')}

Experience:
{profile_sections.get('Experience', '')}

Education:
{profile_sections.get('Education', '')}
"""

STRUCTURED_PROFILE_SCHEMA = {
    "type": "object",
    "properties": {
        "skills": {"type": "array", "items": {"type": "string"}},
        "industry": {"type": "string"},
        "years_of_experience": {"type": "number"},
        "experience_summary": {"type": "string"},
        "highest_education": {"type": "string"},
    },
    "required": ["skills", "industry", "years_of_experience", "experience_summary", "highest_education"],
    "additionalProperties": False,
}

extraction_response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": "Extract a structured profile from the given LinkedIn sections. skills should include both explicitly listed skills and skills clearly implied by the experience/summary text. years_of_experience should be your best estimate total professional years.",
        },
        {"role": "user", "content": extraction_input},
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "structured_profile",
            "schema": STRUCTURED_PROFILE_SCHEMA,
            "strict": True,
        }
    },
)

structured_profile = json.loads(extraction_response.output_text)
structured_profile["name"] = profile_header.get("name", "")
structured_profile["location"] = profile_header.get("location", "")

print(json.dumps(structured_profile, indent=2))

# Project 3, Phase 1: explore the O*NET dataset

**What is O*NET?** It is a free, public database maintained by the US Department of Labor. It describes around 900 occupations (jobs), and for each one it has real, survey collected data: what skills it needs, what a person's interests typically look like if they enjoy that job, what abilities it requires, and more.

**Why are we using it?** Right now our `input/business_ideas.json` has 8 entries we wrote ourselves by hand, our best guess. O*NET gives us real data instead of guesses for at least two important fields: what skills a business idea needs, and what kind of interests/personality suits it. We still have to invent budget and time estimates ourselves (O*NET has no data on that), but two out of several fields being real instead of guessed is a real improvement.

O*NET is split into many files, each one a big table. We are not going to load all of them, only three:
1. **Occupation Data**, the basics: a code, a title, a description. We look at this one first, right now.
2. **Interests**, how much each occupation fits each of 6 interest types (this is what we will use for personality/trait matching)
3. **Skills**, how important each skill is for each occupation

Let's look at file 1 only for now.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("client ready:", client is not None)

In [ ]:
import os
import pandas as pd

ONET_DIR = "input/onet"

occupation_df = pd.read_csv(os.path.join(ONET_DIR, "occupation_data.csv"))

print("shape (rows, columns):", occupation_df.shape)
occupation_df.head(10)

In [ ]:
len(occupation_df)

## Job Zone and software skills

Two more files, these will help reasoning about budget and time more consistently than guessing from a two sentence job description alone.

**Job Zone** (`job_zones.csv`) rates every occupation on how much preparation it needs. O*NET also publishes a plain English explanation of each number (`job_zone_reference.csv`), quick summary (note: in this O*NET version, zones 1 and 2 are merged into one bucket):

| Zone | Meaning | Example jobs |
|---|---|---|
| 1-2 | Very little to some preparation, high school, a few days to a year of training | dishwashers, tellers, customer service reps |
| 3 | Medium preparation, vocational training or associate's degree | electricians, medical assistants |
| 4 | Considerable preparation, usually a bachelor's degree | accountants, graphic designers |
| 5 | Extensive preparation, graduate school, 5+ years experience | lawyers, pharmacists |

**Note:** "Job Zone" is about years of training, not the same thing as startup cost, but it's a decent proxy signal: a Zone 1-2 job usually also needs little equipment to start a small business version of it, a Zone 4 or 5 job usually needs more (credentials, tools, sometimes licensing).

**Software skills** (`software_skills.csv`) lists software used in each occupation, lots of specific brand names (like "Adobe Illustrator"), each with an "Element Name" giving the general category (e.g. "Graphics or photo imaging software"), more useful for us than the brand name.

(This O*NET version dropped the separate "physical tools" file that used to exist alongside this one, software skills is what we have available now.)

In [ ]:
job_zones_df = pd.read_csv(os.path.join(ONET_DIR, "job_zones.csv"))
job_zone_ref_df = pd.read_csv(os.path.join(ONET_DIR, "job_zone_reference.csv"))
software_skills_df = pd.read_csv(os.path.join(ONET_DIR, "software_skills.csv"))

print("job_zones_df shape:", job_zones_df.shape)
print("software_skills_df shape:", software_skills_df.shape)

# example: Web Developers (same occupation we looked at in earlier lab tests)
example_code = "15-1254.00"

zone = job_zones_df.loc[job_zones_df["O*NET-SOC Code"] == example_code, "Job Zone"].values[0]
print(f"\nJob Zone for {example_code} (Web Developers): {zone}")

print("\nSoftware categories (Element Name, deduplicated, first 8):")
print(software_skills_df.loc[software_skills_df["O*NET-SOC Code"] == example_code, "Element Name"].drop_duplicates().head(8).to_string(index=False))

## Interests (RIASEC)

This is the file our trait matching will actually depend on. RIASEC is a psychology model (Holland, 1959) with 6 interest types, and it is exactly what O*NET's `career_interest_types.csv` scores every occupation on, **1 to 7 scale**, same range as our TIPI Big Five scores:

- **R**ealistic, hands on, working with tools, machines, or the outdoors
- **I**nvestigative, analytical, research, solving abstract problems
- **A**rtistic, creative, original, unstructured work
- **S**ocial, helping, teaching, working closely with people
- **E**nterprising, persuading, leading, business/sales minded
- **C**onventional, organized, detail oriented, structured/procedural work

Each occupation gets a score on all 6, not just one, a Web Developer might score high on Investigative and Conventional but low on Social, for example.

The raw file has more rows than just these 6 per occupation (it also has some "which is your 1st/2nd/3rd highest interest" rows we do not need), so the code below filters to just the 6 numeric scores and reshapes it to one row per occupation, one column per interest type, easier to read.

In [ ]:
interests_df = pd.read_csv(os.path.join(ONET_DIR, "career_interest_types.csv"))
print("interests_df shape (raw, before filtering):", interests_df.shape)
print("Scale ID values present:", interests_df["Scale ID"].unique())

riasec_df = interests_df[interests_df["Scale ID"] == "OI"]

riasec_pivot = riasec_df.pivot(index="O*NET-SOC Code", columns="Element Name", values="Data Value")
print("\nriasec_pivot shape (one row per occupation, one column per interest type):", riasec_pivot.shape)

# compare two occupations we already looked at: Web Developers vs Chief Executives
riasec_pivot.loc[["15-1254.00", "11-1011.00"]]

**What "OI" and "IH" mean** (from O*NET's official `scales_reference.csv`:

- **OI** = Occupational Interests, scale 1 to 7, this is the actual RIASEC strength score, what we use
- **IH** = Occupational Interest High-Point, scale 0 to 6, just a code for which interest type ranks 1st/2nd/3rd for that occupation, not a strength score

## Skills (essential + transferable)

Last of the three main files. This O*NET version splits what used to be one "Skills" file into two: `essential_skills.csv` (10 basic skills, e.g. Reading Comprehension, Critical Thinking) and `transferable_skills.csv` (25 more specific ones, e.g. Programming, Negotiation, Troubleshooting). Together that is the same 35 skills as before, we just combine them back into one table.

Each skill has two scores: **IM** (Importance, 1 to 5, how important this skill is for the job) and **LV** (Level, 0 to 7, how much of it is needed). We only need Importance, to pick the top skills for a business idea.

In [ ]:
essential_df = pd.read_csv(os.path.join(ONET_DIR, "essential_skills.csv"))
transferable_df = pd.read_csv(os.path.join(ONET_DIR, "transferable_skills.csv"))

skills_df = pd.concat([essential_df, transferable_df], ignore_index=True)
print("combined skills_df shape:", skills_df.shape)
print("unique skills total:", skills_df["Element Name"].nunique())

importance_df = skills_df[skills_df["Scale ID"] == "IM"]

# top 8 skills by importance for Web Developers
top_skills = (
    importance_df[importance_df["O*NET-SOC Code"] == example_code]
    .sort_values("Data Value", ascending=False)
    .head(8)
)
top_skills[["Element Name", "Data Value"]]

## Work styles and typical work week length

Two more signals, found by checking what else O*NET has beyond the three main files:

- **Work styles** (`work_styles.csv`), 21 personality-like traits per occupation (Innovation, Achievement Orientation, Attention to Detail, Stress Tolerance, Cooperation, etc.), much closer to Big Five than RIASEC. We use the **WI** scale (Work Styles Impact, -3 to 3), higher means more important for that occupation.
- **Duration of Typical Work Week**, one specific field inside `work_context.csv` (we filtered the original 51MB file down to just this field, saved as `work_context_duration.csv`, so we don't load the whole thing). It uses the **CT** scale, a 1 to 3 average (roughly 1 = usually under 40h/week, 3 = usually over 40h/week), a real signal for how time intensive the work tends to be, instead of guessing purely from the job description.

In [ ]:
work_styles_df = pd.read_csv(os.path.join(ONET_DIR, "work_styles.csv"))
work_context_duration_df = pd.read_csv(os.path.join(ONET_DIR, "work_context_duration.csv"))

work_styles_wi_df = work_styles_df[work_styles_df["Scale ID"] == "WI"]

top_work_styles = (
    work_styles_wi_df[work_styles_wi_df["O*NET-SOC Code"] == example_code]
    .sort_values("Data Value", ascending=False)
    .head(5)
)
print("Top work styles for Web Developers:")
print(top_work_styles[["Element Name", "Data Value"]].to_string(index=False))

duration_score = work_context_duration_df.loc[
    (work_context_duration_df["O*NET-SOC Code"] == example_code) & (work_context_duration_df["Scale ID"] == "CT"),
    "Data Value",
].values[0]
print(f"\nDuration of typical work week (1=under 40h, 2=40h, 3=over 40h) for Web Developers: {duration_score:.2f}")

## QA finding: ideal_traits were not actually differentiated

Checking the first batch of 20 ideas, 7 of them came back with the exact same `ideal_traits` (openness [4,7], conscientiousness [5,7], extraversion [3,5], agreeableness [4,6], neuroticism [2,4]), including occupations as different as Web Developers and Carpenters. Their real `top_work_styles` are clearly different (Web Developers: Innovation, Intellectual Curiosity; Carpenters: Cautiousness, Perseverance, Integrity, no openness related styles at all), but the LLM defaulted to a generic "safe professional" range instead of using that signal. This made `trait_fit`, our biggest weight (35%), fail to discriminate between many ideas, so unrelated ones could rank highly just from small noise in the other scores.

**Fix:** compute a numeric anchor per Big Five trait from the real `top_work_styles`, deterministic, not LLM invented, then require the enrichment step to build its range around that anchor instead of free inventing one.

In [ ]:
WORK_STYLE_TRAIT_BOOST = {
    "Innovation": ("openness", 1), "Achievement Orientation": ("conscientiousness", 1),
    "Intellectual Curiosity": ("openness", 1), "Tolerance for Ambiguity": ("openness", 1),
    "Initiative": ("extraversion", 1), "Adaptability": ("neuroticism", -1),
    "Self-Confidence": ("neuroticism", -1), "Perseverance": ("conscientiousness", 1),
    "Leadership Orientation": ("extraversion", 1), "Humility": ("agreeableness", 1),
    "Sincerity": ("agreeableness", 1), "Empathy": ("agreeableness", 1),
    "Cooperation": ("agreeableness", 1), "Optimism": ("neuroticism", -1),
    "Social Orientation": ("extraversion", 1), "Cautiousness": ("conscientiousness", 1),
    "Attention to Detail": ("conscientiousness", 1), "Dependability": ("conscientiousness", 1),
    "Integrity": ("conscientiousness", 1), "Stress Tolerance": ("neuroticism", -1),
    "Self-Control": ("conscientiousness", 1),
}

def compute_trait_anchors(top_work_styles: list) -> dict:
    anchors = {t: 4.0 for t in ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]}
    for style in top_work_styles:
        if style in WORK_STYLE_TRAIT_BOOST:
            trait, direction = WORK_STYLE_TRAIT_BOOST[style]
            anchors[trait] += direction * 0.6
    return {t: round(max(1.0, min(7.0, v)), 1) for t, v in anchors.items()}

# sanity check: Web Developers vs Carpenters should now look meaningfully different
web_dev_styles = ["Attention to Detail", "Innovation", "Dependability", "Intellectual Curiosity", "Adaptability"]
carpenter_styles = ["Dependability", "Attention to Detail", "Cautiousness", "Perseverance", "Integrity"]
print("Web Developers anchor:", compute_trait_anchors(web_dev_styles))
print("Carpenters anchor:    ", compute_trait_anchors(carpenter_styles))

## Pick occupations and combine everything into one record each

Now we combine everything we explored (title, description, RIASEC, Job Zone, top skills, top work styles, typical work week length) into one clean record per occupation. This becomes the raw input for the next step, one LLM call per occupation that turns it into an actual business idea (name, budget estimate, time estimate, ideal Big Five traits).

I picked 20 occupations spanning different RIASEC profiles, deliberately ones accessible to a solo person with a small budget (web development, tutoring, personal training, bookkeeping, writing, cleaning, handyman work, HR/admin support, coaching, art), avoiding ones that need heavy licensing (medicine, law).

In [ ]:
OCCUPATION_CODES = [
    "15-1254.00",  # Web Developers
    "27-1024.00",  # Graphic Designers
    "39-9031.00",  # Exercise Trainers and Group Fitness Instructors
    "25-3041.00",  # Tutors
    "35-2014.00",  # Cooks, Restaurant
    "39-5012.00",  # Hairdressers, Hairstylists, and Cosmetologists
    "13-2011.00",  # Accountants and Auditors
    "27-3043.00",  # Writers and Authors
    "27-1026.00",  # Merchandise Displayers and Window Trimmers
    "39-9011.00",  # Childcare Workers
    "37-2012.00",  # Maids and Housekeeping Cleaners
    "47-2031.00",  # Carpenters
    "13-1161.00",  # Market Research Analysts and Marketing Specialists
    "27-1022.00",  # Fashion Designers
    "15-1211.00",  # Computer Systems Analysts
    "27-2022.00",  # Coaches and Scouts
    "13-1071.00",  # Human Resources Specialists
    "43-6014.00",  # Secretaries and Administrative Assistants
    "15-1232.00",  # Computer User Support Specialists
    "27-1013.00",  # Fine Artists
]

def build_occupation_record(code: str) -> dict:
    occ_row = occupation_df.loc[occupation_df["O*NET-SOC Code"] == code].iloc[0]
    riasec = riasec_pivot.loc[code].to_dict()
    zone = job_zones_df.loc[job_zones_df["O*NET-SOC Code"] == code, "Job Zone"].values[0]
    top_skill_rows = (
        importance_df[importance_df["O*NET-SOC Code"] == code]
        .sort_values("Data Value", ascending=False)
        .head(5)
    )
    top_work_style_rows = (
        work_styles_wi_df[work_styles_wi_df["O*NET-SOC Code"] == code]
        .sort_values("Data Value", ascending=False)
        .head(5)
    )
    duration_rows = work_context_duration_df.loc[
        (work_context_duration_df["O*NET-SOC Code"] == code) & (work_context_duration_df["Scale ID"] == "CT"),
        "Data Value",
    ]
    top_work_styles = top_work_style_rows["Element Name"].tolist()
    return {
        "code": code,
        "title": occ_row["Title"],
        "description": occ_row["Description"],
        "riasec": riasec,
        "job_zone": int(zone),
        "top_skills": top_skill_rows["Element Name"].tolist(),
        "top_work_styles": top_work_styles,
        "trait_anchors": compute_trait_anchors(top_work_styles),
        "typical_work_week_score": float(duration_rows.values[0]) if len(duration_rows) else None,
    }

occupation_records = [build_occupation_record(code) for code in OCCUPATION_CODES]
print(len(occupation_records), "occupation records built")
occupation_records[0]

## Turn each occupation into a business idea

One LLM call per occupation (20 calls total, cheap with `gpt-4o-mini`, a few cents and about a minute to run). The model only invents what O*NET does not give us: the business framing, the budget estimate, the time estimate, and the ideal Big Five ranges (guided by the real RIASEC scores). Everything else, `skills_needed` and the grounding data, comes from real O*NET fields, and we save the O*NET code in `source_note` so every entry is traceable back to where it came from.

Same schema as the original hand written `input/business_ideas.json`, so this new file is a drop in replacement later.

In [ ]:
IDEA_SCHEMA = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "description": {"type": "string"},
        "category": {"type": "string"},
        "budget_range_eur": {"type": "array", "items": {"type": "number"}, "minItems": 2, "maxItems": 2},
        "time_range_hours_per_week": {"type": "array", "items": {"type": "number"}, "minItems": 2, "maxItems": 2},
        "risk_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "skills_needed": {"type": "array", "items": {"type": "string"}},
        "ideal_traits": {
            "type": "object",
            "properties": {
                "openness": {"type": "array", "items": {"type": "number"}, "minItems": 2, "maxItems": 2},
                "conscientiousness": {"type": "array", "items": {"type": "number"}, "minItems": 2, "maxItems": 2},
                "extraversion": {"type": "array", "items": {"type": "number"}, "minItems": 2, "maxItems": 2},
                "agreeableness": {"type": "array", "items": {"type": "number"}, "minItems": 2, "maxItems": 2},
                "neuroticism": {"type": "array", "items": {"type": "number"}, "minItems": 2, "maxItems": 2},
            },
            "required": ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"],
            "additionalProperties": False,
        },
    },
    "required": ["name", "description", "category", "budget_range_eur", "time_range_hours_per_week", "risk_level", "skills_needed", "ideal_traits"],
    "additionalProperties": False,
}

ENRICHMENT_SYSTEM_PROMPT = (
    "Turn the given O*NET occupation into a small, low cost business idea a solo person could start, not a job posting. "
    "name: a short business idea name (e.g. 'Freelance Web Development Service', not the occupation title itself). "
    "description: 1 to 2 sentences describing the business, not the job. "
    "category: a short category label. "
    "budget_range_eur: realistic typical startup budget range in EUR for a solo person starting this small, [min, max], "
    "use the job_zone and top_skills given (more preparation/specialized software usually means a higher budget). "
    "time_range_hours_per_week: realistic typical time commitment range to run this as a side business or small operation, [min, max], "
    "use job_zone and typical_work_week_score as signals (typical_work_week_score is 1 to 3, 1 means the full time version of this "
    "job usually takes under 40h/week, 3 means usually over 40h/week, a higher score suggests this type of work tends to be more "
    "time intensive even in a smaller version). "
    "risk_level: low, medium, or high. "
    "skills_needed: 3 to 5 practical skills, simplified from the given top_skills list, plain language. "
    "ideal_traits: Big Five ranges on a 1 to 7 scale, one [min, max] range per trait. The user message gives you trait_anchors, a "
    "pre-computed numeric estimate (1 to 7) for each trait, already derived from this occupation's real top_work_styles data. "
    "You MUST center each trait's range on its trait_anchor value, roughly anchor minus 1.5 to anchor plus 1.5 (clipped to 1-7), "
    "do NOT default to a generic middle of scale range like 4-7 or 5-7 for every occupation, the anchors are already different per "
    "occupation because the real work style data is different, your range must reflect that difference. Only deviate from an anchor "
    "if the occupation's specific description or skills give a clear concrete reason to."
)

enriched_ideas = []
for i, occ in enumerate(occupation_records, start=1):
    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": ENRICHMENT_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(occ)},
        ],
        text={"format": {"type": "json_schema", "name": "business_idea", "schema": IDEA_SCHEMA, "strict": True}},
    )
    idea = json.loads(response.output_text)
    idea["id"] = f"onet_{i:03d}"
    idea["source_note"] = f"Derived from O*NET occupation {occ['code']} ({occ['title']}), real RIASEC/skills/work styles data, budget/time/traits are LLM estimates"
    enriched_ideas.append(idea)
    print(f"{i}/{len(occupation_records)}: {idea['name']}")

print(f"\n{len(enriched_ideas)} business ideas enriched")

In [ ]:
ENRICHED_IDEAS_PATH = os.path.join(ONET_DIR, "business_ideas_enriched.json")

with open(ENRICHED_IDEAS_PATH, "w") as f:
    json.dump(enriched_ideas, f, indent=2)

print("Saved to", ENRICHED_IDEAS_PATH)
enriched_ideas[0]

## QA fix: recalibrate 2 budget outliers

`onet_010` (Mobile Childcare) and `onet_011` (Home Cleaning) came back at €2000-5000, clearly above the rest of the batch (mostly €200-1500). Re-running just these two with a calibration hint added to the prompt, showing the model the budget range the other 18 ideas landed in, so it estimates relative to that instead of in isolation.

In [ ]:
RECALIBRATE_IDS = ["onet_010", "onet_011"]

other_budgets = [
    idea["budget_range_eur"] for idea in enriched_ideas if idea["id"] not in RECALIBRATE_IDS
]
budget_floor = min(b[0] for b in other_budgets)
budget_ceiling = max(b[1] for b in other_budgets)

CALIBRATION_HINT = (
    f"\n\nCalibration note: this occupation is part of a batch of similar solo, low cost business ideas. "
    f"The other ideas in the batch have budget_range_eur values between EUR {budget_floor} and EUR {budget_ceiling}. "
    f"Estimate this one relative to that range, only go noticeably higher if there is a specific, concrete cost driver "
    f"(for example required certification, insurance, or specialized equipment), and if so keep it as close to that "
    f"range as the real cost driver allows, do not inflate the estimate without a concrete reason."
)

id_to_index = {idea["id"]: i for i, idea in enumerate(enriched_ideas)}
code_to_index = {code: i for i, code in enumerate(OCCUPATION_CODES)}

for target_id in RECALIBRATE_IDS:
    idx = id_to_index[target_id]
    occ = occupation_records[idx]  # same position in both lists, ids assigned in order

    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": ENRICHMENT_SYSTEM_PROMPT + CALIBRATION_HINT},
            {"role": "user", "content": json.dumps(occ)},
        ],
        text={"format": {"type": "json_schema", "name": "business_idea", "schema": IDEA_SCHEMA, "strict": True}},
    )
    idea = json.loads(response.output_text)
    idea["id"] = target_id
    idea["source_note"] = f"Derived from O*NET occupation {occ['code']} ({occ['title']}), real RIASEC/skills/work styles data, budget/time/traits are LLM estimates"
    enriched_ideas[idx] = idea
    print(f"{target_id}: {idea['name']} -> budget {idea['budget_range_eur']} EUR, time {idea['time_range_hours_per_week']} h/wk, risk {idea['risk_level']}")

## Embed the business ideas with Cohere

For each idea, we build one text combining name, description, category, and skills, that is what gets embedded (turned into a vector of 1536 numbers). Later, when a user submits their profile, we embed a text built the same way from their skills/traits, and Pinecone finds the closest business idea vectors, semantic search, instead of the plain Python loop we used in the lab MVP.

Cohere's `embed-v4.0` model needs an `input_type`: `search_document` when embedding the things you will search over (our business ideas), `search_query` when embedding the thing doing the searching (the user's profile, later). Same model, different mode, so the two vector spaces line up correctly.

In [ ]:
import cohere

co = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

def build_idea_embedding_text(idea: dict) -> str:
    return (
        f"{idea['name']}. {idea['description']} "
        f"Category: {idea['category']}. "
        f"Skills needed: {', '.join(idea['skills_needed'])}."
    )

idea_texts = [build_idea_embedding_text(idea) for idea in enriched_ideas]

embed_response = co.embed(
    texts=idea_texts,
    model="embed-v4.0",
    input_type="search_document",
    embedding_types=["float"],
)
idea_embeddings = embed_response.embeddings.float_

print(f"{len(idea_embeddings)} embeddings computed, dimension {len(idea_embeddings[0])}")
print()
print("Example text for onet_001:")
print(idea_texts[0])

## Create the Pinecone index

Serverless index, dimension 1536 (matches `embed-v4.0`), cosine similarity (standard for text embeddings). This checks first and only creates it if it does not already exist yet, so re-running this cell later is safe.

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import time

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

PINECONE_INDEX_NAME = "entrepreneur-coach-ideas"

if PINECONE_INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(1)
    print(f"Created index '{PINECONE_INDEX_NAME}'")
else:
    print(f"Index '{PINECONE_INDEX_NAME}' already exists")

index = pc.Index(PINECONE_INDEX_NAME)
print(index.describe_index_stats())

## Upsert the ideas into Pinecone

Pinecone metadata only accepts flat values (strings, numbers, booleans, lists of strings), not nested objects, so `ideal_traits` (a dict of ranges) can't go in as-is. Simplest fix: store the whole idea as one JSON string under a `data` key, on retrieval we just `json.loads()` it back to the exact same dict shape our fit calculation already expects.

In [ ]:
vectors = [
    {
        "id": idea["id"],
        "values": embedding,
        "metadata": {"data": json.dumps(idea), "name": idea["name"], "category": idea["category"]},
    }
    for idea, embedding in zip(enriched_ideas, idea_embeddings)
]

index.upsert(vectors=vectors)

time.sleep(2)  # let the index finish updating its stats
print(index.describe_index_stats())

## Test retrieval

Embed a sample query with `input_type="search_query"` (not `search_document`, that matters for Cohere to line up the vector spaces correctly), then ask Pinecone for the closest matches. This is the same mechanism the real app will use, just with a made up query for now instead of a real user profile.

In [ ]:
test_query_text = "Data engineer with programming, data modeling, and problem solving skills, high conscientiousness and openness, moderate budget"

query_embed_response = co.embed(
    texts=[test_query_text],
    model="embed-v4.0",
    input_type="search_query",
    embedding_types=["float"],
)
query_embedding = query_embed_response.embeddings.float_[0]

results = index.query(vector=query_embedding, top_k=5, include_metadata=True)

for match in results["matches"]:
    print(f"{match['score']:.3f}  {match['id']}  {match['metadata']['name']}")

# Project 3, Phase 2: LangGraph pipeline

Until now, `pipeline.py` (and `app.py`) call the steps as plain Python functions, one after another. `PROJECT_PLAN.md` commits to LangGraph specifically instead of a plain function chain, because the flow here is fixed (not something an agent should decide on its own) and each step's state should be inspectable on its own, useful for debugging and for showing the graph structure in the presentation.

**Design choice:** the app has a natural pause in the middle, the user only picks which idea to build a 90 day roadmap for after seeing the ranked list, `app.py` already reflects this with two separate button clicks. Instead of one graph with an `interrupt()` (which needs a checkpointer and a thread id to resume later, more moving parts than this project needs), we build **two graphs**:

1. **Recommendation graph**: score the Big Five, parse the PDF, extract the structured profile, retrieve candidates from Pinecone, rank them by career best fit. Ends with the ranked list.
2. **Coaching graph**: takes the ranked list plus the idea the user picked, writes the report narrative, exports it as PDF and HTML. This is the "explain" step from the plan.

Both graphs reuse the exact functions already in `pipeline.py`, we are not rewriting any logic here, just wiring the same functions into LangGraph nodes so the flow is an actual graph instead of a chain of function calls. Once this is confirmed working, `app.py` can call `recommendation_graph.invoke(...)` / `coaching_graph.invoke(...)` instead of calling the `pipeline.py` functions directly.

In [ ]:
from typing import TypedDict, Optional

from langgraph.graph import StateGraph, END

import pipeline

print("langgraph and pipeline ready")

## Recommendation graph

State holds everything from the raw inputs (PDF path, TIPI answers, budget, time) through to the final ranked list. Each node reads what it needs from the state and returns only the new keys it adds, LangGraph merges that into the state automatically, so nodes stay small and each one's output is inspectable on its own (print `recommendation_result["structured_profile"]`, or `["big_five_scores"]`, etc, separately, instead of only seeing one final printout).

In [ ]:
class RecommendationState(TypedDict):
    pdf_path: str
    tipi_answers: dict
    budget_eur: float
    time_available_hours_per_week: float
    profile_sections: dict
    profile_header: dict
    structured_profile: dict
    big_five_scores: dict
    candidate_ideas: list
    grounded_top_ideas: list


def node_score_personality(state: RecommendationState) -> dict:
    big_five_scores = pipeline.score_tipi(state["tipi_answers"])
    return {"big_five_scores": big_five_scores}


def node_parse_pdf(state: RecommendationState) -> dict:
    profile_sections, profile_header = pipeline.parse_profile_pdf(state["pdf_path"])
    return {"profile_sections": profile_sections, "profile_header": profile_header}


def node_extract_profile(state: RecommendationState) -> dict:
    structured_profile = pipeline.extract_structured_profile(client, state["profile_sections"], state["profile_header"])
    return {"structured_profile": structured_profile}


def node_retrieve_candidates(state: RecommendationState) -> dict:
    candidate_ideas = pipeline.retrieve_candidate_ideas(state["structured_profile"])
    return {"candidate_ideas": candidate_ideas}


def node_rank_by_fit(state: RecommendationState) -> dict:
    skill_fits = pipeline.compute_skill_fits(state["structured_profile"]["skills"], state["candidate_ideas"])
    ranked = sorted(
        (
            pipeline.compute_career_best_fit(
                idea, state["structured_profile"], state["big_five_scores"],
                state["budget_eur"], state["time_available_hours_per_week"], skill_fits[idea["id"]],
            )
            for idea in state["candidate_ideas"]
        ),
        key=lambda r: r["career_best_fit_percentage"],
        reverse=True,
    )
    top = ranked[:5]
    ideas_by_id = {idea["id"]: idea for idea in state["candidate_ideas"]}
    grounded_top_ideas = [
        {
            **r,
            "matched_skills": pipeline.matched_skills(state["structured_profile"]["skills"], ideas_by_id[r["id"]]["skills_needed"]),
            "in_range_traits": pipeline.in_range_traits(ideas_by_id[r["id"]], state["big_five_scores"]),
        }
        for r in top
    ]
    return {"grounded_top_ideas": grounded_top_ideas}


recommendation_builder = StateGraph(RecommendationState)
recommendation_builder.add_node("score_personality", node_score_personality)
recommendation_builder.add_node("parse_pdf", node_parse_pdf)
recommendation_builder.add_node("extract_profile", node_extract_profile)
recommendation_builder.add_node("retrieve_candidates", node_retrieve_candidates)
recommendation_builder.add_node("rank_by_fit", node_rank_by_fit)

recommendation_builder.set_entry_point("score_personality")
recommendation_builder.add_edge("score_personality", "parse_pdf")
recommendation_builder.add_edge("parse_pdf", "extract_profile")
recommendation_builder.add_edge("extract_profile", "retrieve_candidates")
recommendation_builder.add_edge("retrieve_candidates", "rank_by_fit")
recommendation_builder.add_edge("rank_by_fit", END)

recommendation_graph = recommendation_builder.compile()
print("recommendation_graph compiled")

Run it, same test profile PDF as the earlier steps, a neutral TIPI test set (mostly leaning open/conscientious/calm, matching a data-oriented profile), and the same budget/time we used in earlier manual tests.

In [ ]:
test_tipi_answers = {1: 5, 2: 3, 3: 6, 4: 3, 5: 6, 6: 3, 7: 5, 8: 2, 9: 5, 10: 2}

recommendation_result = recommendation_graph.invoke({
    "pdf_path": PROFILE_PDF_PATH,
    "tipi_answers": test_tipi_answers,
    "budget_eur": 800,
    "time_available_hours_per_week": 12,
})

print("structured_profile:", json.dumps(recommendation_result["structured_profile"], indent=2))
print()
print("big_five_scores:", recommendation_result["big_five_scores"])
print()
for r in recommendation_result["grounded_top_ideas"]:
    print(f"{r['career_best_fit_percentage']}%  {r['id']}  {r['name']}")

## Coaching graph: explain

Second graph, picks up where the recommendation graph left off. Input state is the ranked ideas plus which one the user picked (`roadmap_idea_id`), output is the written report narrative and the two exported files.

In [ ]:
class CoachingState(TypedDict):
    structured_profile: dict
    big_five_scores: dict
    tipi_answers: dict
    budget_eur: float
    time_available_hours_per_week: float
    grounded_top_ideas: list
    roadmap_idea_id: Optional[str]
    report_narrative: dict
    pdf_path: str
    html_path: str


def node_generate_report(state: CoachingState) -> dict:
    report_narrative = pipeline.generate_output_report(
        client, state["structured_profile"], state["big_five_scores"],
        state["budget_eur"], state["time_available_hours_per_week"],
        state["grounded_top_ideas"], state.get("roadmap_idea_id"),
    )
    return {"report_narrative": report_narrative}


def node_export_pdf(state: CoachingState) -> dict:
    pdf_path = pipeline.export_report_pdf(
        state["structured_profile"], state["report_narrative"], state["grounded_top_ideas"],
        "output/entrepreneur_coach_report.pdf", state.get("roadmap_idea_id"), state.get("tipi_answers"),
    )
    return {"pdf_path": pdf_path}


def node_export_html(state: CoachingState) -> dict:
    html_path = pipeline.export_report_html(
        state["structured_profile"], state["report_narrative"], state["grounded_top_ideas"],
        "output/entrepreneur_coach_report.html", state.get("roadmap_idea_id"), state.get("tipi_answers"),
    )
    return {"html_path": html_path}


coaching_builder = StateGraph(CoachingState)
coaching_builder.add_node("generate_report", node_generate_report)
coaching_builder.add_node("export_pdf", node_export_pdf)
coaching_builder.add_node("export_html", node_export_html)

coaching_builder.set_entry_point("generate_report")
coaching_builder.add_edge("generate_report", "export_pdf")
coaching_builder.add_edge("export_pdf", "export_html")
coaching_builder.add_edge("export_html", END)

coaching_graph = coaching_builder.compile()
print("coaching_graph compiled")

In [ ]:
top_idea_id = recommendation_result["grounded_top_ideas"][0]["id"]

coaching_result = coaching_graph.invoke({
    "structured_profile": recommendation_result["structured_profile"],
    "big_five_scores": recommendation_result["big_five_scores"],
    "tipi_answers": recommendation_result["tipi_answers"],
    "budget_eur": recommendation_result["budget_eur"],
    "time_available_hours_per_week": recommendation_result["time_available_hours_per_week"],
    "grounded_top_ideas": recommendation_result["grounded_top_ideas"],
    "roadmap_idea_id": top_idea_id,
})

print("report for:", top_idea_id)
print("pdf_path:", coaching_result["pdf_path"])
print("html_path:", coaching_result["html_path"])

Both graphs use the exact same functions from `pipeline.py`, this only changes how they are called, not what they do, ranking and report numbers should come out identical to running `pipeline.py` directly.

# Project 3, Phase 1 continued: scale the business idea dataset

Right now we only have 20 business ideas, `RETRIEVAL_TOP_K=10` pulls half the whole dataset as candidates every time, not very selective. We agreed to scale to the full filtered pool of O*NET occupations instead of hand picking more one by one.

**Filter used to pick the candidate pool**, same reasoning as picking the original 20 by hand (solo, low budget business, no heavy licensing needed), just automated this time:
- **Job Zone below 5**, excludes occupations needing a graduate degree or 5+ years experience (doctors, lawyers, scientists)
- **Major occupation group** limited to 13 groups that translate reasonably to a solo business (Management, Business/Financial, Computer/Math, Engineering, Science, Community/Social Service, Education, Arts/Design, Personal Care, Sales, Office/Admin, Construction, Installation/Repair), excluding groups that are mostly employed only roles or need licensing (Legal, Healthcare Practitioners, Healthcare Support, Protective Service, Military, Farming/Fishing/Forestry, Production, Transportation, Food Prep, Building/Grounds Cleaning)

This gives 490 candidates. We keep the original 20 on top of that (2 of them, Cooks and Housekeeping Cleaners, technically fall in an excluded group, but they already produced good ideas, so no reason to drop them), for a combined dataset of 492 business ideas.

In [ ]:
merged = occupation_df.merge(job_zones_df[["O*NET-SOC Code", "Job Zone"]], on="O*NET-SOC Code", how="left")
merged["major_group"] = merged["O*NET-SOC Code"].str[:2]

KEEP_MAJOR_GROUPS = ["11", "13", "15", "17", "19", "21", "25", "27", "39", "41", "43", "47", "49"]
# excluded: 23 Legal, 29 Healthcare Practitioners, 31 Healthcare Support, 33 Protective Service, 55 Military,
# 45 Farming/Fishing/Forestry, 51 Production, 53 Transportation, 35 Food Prep, 37 Building/Grounds Cleaning
# same reasoning as picking the original 20 by hand: solo, low budget business, no heavy licensing needed

filtered = merged[(merged["major_group"].isin(KEEP_MAJOR_GROUPS)) & (merged["Job Zone"] < 5)]
filtered_codes = set(filtered["O*NET-SOC Code"])

all_target_codes = filtered_codes | set(OCCUPATION_CODES)  # keep the original 20, even the 2 outside the filter
new_codes = sorted(all_target_codes - set(OCCUPATION_CODES))

print("total target dataset size:", len(all_target_codes))
print("new occupations to enrich:", len(new_codes))

## Enrich the new occupations

Reuses `build_occupation_record`, `ENRICHMENT_SYSTEM_PROMPT`, and `IDEA_SCHEMA` from the original 20, exact same process, just more of them. One `gpt-4o-mini` call per occupation, sequential, this is the slow part, expect roughly 15-25 minutes for ~472 calls. Prints progress every 25 so you can see it moving.

In [ ]:
new_occupation_records = [build_occupation_record(code) for code in new_codes]
print(len(new_occupation_records), "new occupation records built")

new_enriched_ideas = []
next_id_number = len(enriched_ideas) + 1  # continue numbering after the existing 20 (onet_021 onward)

for i, occ in enumerate(new_occupation_records):
    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": ENRICHMENT_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(occ)},
        ],
        text={"format": {"type": "json_schema", "name": "business_idea", "schema": IDEA_SCHEMA, "strict": True}},
    )
    idea = json.loads(response.output_text)
    idea["id"] = f"onet_{next_id_number + i:03d}"
    idea["source_note"] = f"Derived from O*NET occupation {occ['code']} ({occ['title']}), real RIASEC/skills/work styles data, budget/time/traits are LLM estimates"
    new_enriched_ideas.append(idea)
    if (i + 1) % 25 == 0 or (i + 1) == len(new_occupation_records):
        print(f"{i + 1}/{len(new_occupation_records)} done, latest: {idea['name']}")

print(f"\n{len(new_enriched_ideas)} new business ideas enriched")

In [ ]:
all_enriched_ideas = enriched_ideas + new_enriched_ideas
print("total business ideas:", len(all_enriched_ideas))

with open(ENRICHED_IDEAS_PATH, "w") as f:
    json.dump(all_enriched_ideas, f, indent=2)

print("saved to", ENRICHED_IDEAS_PATH)

## QA: check for budget outliers automatically

Last time (the original 20) we caught 2 budget outliers by eyeballing the printed list, not realistic at 492 items. Instead, flag anything more than 3x away from the median budget midpoint, a simple statistical check, then look at what comes up. This only flags, it does not auto fix anything, we will decide together whether the flagged ones need the same calibration hint fix as before.

In [ ]:
import statistics

budget_midpoints = [(sum(idea["budget_range_eur"]) / 2, idea) for idea in all_enriched_ideas]
median_midpoint = statistics.median(m for m, _ in budget_midpoints)

outliers = [(m, idea) for m, idea in budget_midpoints if m > median_midpoint * 3 or m < median_midpoint / 3]

print(f"median budget midpoint: EUR {median_midpoint:.0f}")
print(f"{len(outliers)} potential outliers (more than 3x away from the median):\n")
for m, idea in sorted(outliers, key=lambda x: -x[0]):
    print(f"{idea['id']}  {idea['name']:<45}  budget {idea['budget_range_eur']}  (midpoint {m:.0f})")

## Drop occupations that don't fit the use case

Not a budget calibration problem, these specific ideas are heavy industry/specialist credential fields (mining, petroleum, nuclear, demolition) or subject matter that does not fit a "solo, low budget business" recommender (sports betting), regardless of what budget number they got. Checking the "Mini Excavation Services" duplicate first, then dropping the 9 ideas that don't belong, then re-saving the JSON before moving on to embeddings.

In [ ]:
for idea_id in ["onet_397", "onet_434"]:
    idea = next(i for i in all_enriched_ideas if i["id"] == idea_id)
    print(idea_id, "->", idea["source_note"])
    print(" ", idea["description"])
    print()

EXCLUDE_IDS = ["onet_167", "onet_168", "onet_169", "onet_186", "onet_289", "onet_436", "onet_439", "onet_441", "onet_486"]

all_enriched_ideas = [idea for idea in all_enriched_ideas if idea["id"] not in EXCLUDE_IDS]
print(f"dropped {len(EXCLUDE_IDS)} ideas, {len(all_enriched_ideas)} remain")

with open(ENRICHED_IDEAS_PATH, "w") as f:
    json.dump(all_enriched_ideas, f, indent=2)

print("re-saved to", ENRICHED_IDEAS_PATH)

## Re-embed everything with Cohere

Cohere's embed endpoint accepts at most 96 texts per call, we have 492, so this batches in chunks of 90 and concatenates the results. Same `build_idea_embedding_text` and `input_type="search_document"` as before, nothing new here beyond doing it in batches.

In [ ]:
def chunked(items: list, size: int):
    for i in range(0, len(items), size):
        yield items[i:i + size]


all_idea_texts = [build_idea_embedding_text(idea) for idea in all_enriched_ideas]

all_idea_embeddings = []
for batch in chunked(all_idea_texts, 90):
    embed_response = co.embed(
        texts=batch,
        model="embed-v4.0",
        input_type="search_document",
        embedding_types=["float"],
    )
    all_idea_embeddings.extend(embed_response.embeddings.float_)

print(f"{len(all_idea_embeddings)} embeddings computed, dimension {len(all_idea_embeddings[0])}")

## Re-upsert everything into Pinecone

Same `id` scheme as before (`onet_001` to `onet_492`), so this overwrites the existing 20 vectors with themselves (harmless) and adds the 472 new ones. Batched at 100 vectors per call, Pinecone's recommended upsert batch size.

In [ ]:
all_vectors = [
    {
        "id": idea["id"],
        "values": embedding,
        "metadata": {"data": json.dumps(idea), "name": idea["name"], "category": idea["category"]},
    }
    for idea, embedding in zip(all_enriched_ideas, all_idea_embeddings)
]

for batch in chunked(all_vectors, 100):
    index.upsert(vectors=batch)

time.sleep(2)  # let the index finish updating its stats
print(index.describe_index_stats())

## Retest retrieval at scale

Same test query and `query_embedding` as the earlier "Test retrieval" cell, now against the full 492 item index instead of 20. `top_k=10` out of 492 is a real filter now (about 2%), instead of pulling half the whole dataset like before.

In [ ]:
results = index.query(vector=query_embedding, top_k=10, include_metadata=True)

print("total vectors in index:", index.describe_index_stats()["total_vector_count"])
print()
for match in results["matches"]:
    print(f"{match['score']:.3f}  {match['id']}  {match['metadata']['name']}")

## Drop exact-name duplicates

Found via a user test (two "Cybersecurity Consulting Service" entries showed up in one ranked list): 19 names are duplicated across 39 of the 483 ideas, closely related O*NET occupations (e.g. Information Security Analysts vs Information Security Engineers) independently landed on the same generic business name, since each occupation is enriched with no awareness of what name the others already picked.

Fix: for each duplicate name, keep only the first one enriched, drop the rest. No re-embedding needed for the ones we keep, their vectors do not change, so this only deletes the dropped ids from Pinecone instead of re-running Cohere on everything again.

In [55]:
seen_names = set()
deduped_ideas = []
removed_ids = []

for idea in all_enriched_ideas:
    if idea["name"] in seen_names:
        removed_ids.append(idea["id"])
    else:
        seen_names.add(idea["name"])
        deduped_ideas.append(idea)

all_enriched_ideas = deduped_ideas
print(f"dropped {len(removed_ids)} exact-name duplicates, {len(all_enriched_ideas)} ideas remain")

with open(ENRICHED_IDEAS_PATH, "w") as f:
    json.dump(all_enriched_ideas, f, indent=2)
print("re-saved to", ENRICHED_IDEAS_PATH)

index.delete(ids=removed_ids)
time.sleep(2)
print(index.describe_index_stats())

dropped 20 exact-name duplicates, 463 ideas remain
re-saved to input/onet/business_ideas_enriched.json
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 463}},
 'total_vector_count': 463,
 'vector_type': 'dense'}
